# Self-Consistency [Step 03.02]

> **MLCourse - Agentic AI - Agent Patterns**

The simplest way to spend more money for more accuracy:

```
                    +--> reasoning path 1 --> 198
                    |
   question --------+--> reasoning path 2 --> 198   ==> MAJORITY: 198
                    |
                    +--> reasoning path 3 --> 270
```

Sample the same question N times at a non-zero temperature, extract the final
answer from each, and take the most common one. That is the whole algorithm
(Wang et al., 2022).

### What you'll learn

- Self-consistency implemented in about ten lines.
- **Measured** accuracy at N=1 vs N=3 vs N=5 on a six-task set.
- Why the *vote spread* is a free confidence signal, and how to use it.
- The cases where it cannot help at all.

### Why it matters

It is the highest ratio of benefit to implementation effort in this whole track -
no framework, no state, no orchestration, about ten lines. It is also a good
lesson in honest evaluation, because on an easy task set it buys you **nothing**
and costs you 5x, and you will only find that out if you measure.

### Prerequisites

- [01_temperature_and_variance](01_temperature_and_variance.ipynb)

### Setup: environment, model, token counting, rate-limit-aware call helper


In [ ]:
import os                              # environment variables
import time                            # timing and pacing
import json                            # pretty-printing structured context
from pathlib import Path               # locating the track root
from dotenv import load_dotenv         # reads KEY=value pairs from .env

# Walk UP from the notebook folder until we hit the repo root, then load the
# (gitignored) .env that lives inside 03_agentic_ai. Note the extra path
# segment: the walk-up lands on the REPO ROOT, not on the track folder.
TRACK = Path.cwd()
while not (TRACK / "03_agentic_ai").exists() and TRACK != TRACK.parent:
    TRACK = TRACK.parent
load_dotenv(TRACK / "03_agentic_ai" / ".env")

GROQ_MODEL = "qwen/qwen3.8-27b"        # hosted, fast, generous free tier
# Local alternative (documented, not used here): Ollama `llama3.1:8b` via
# `from langchain_ollama import ChatOllama`. OpenAI is never used in this course.

from langchain_groq import ChatGroq


def make_llm(temperature: float = 0.0, max_tokens: int = 300, **kw):
    """One place that constructs the chat model, so every notebook is identical."""
    return ChatGroq(model=GROQ_MODEL, temperature=temperature,
                    max_tokens=max_tokens, **kw)


# --- Token counting -----------------------------------------------------------
# Two different numbers, and it matters which one you are looking at:
#   * approx_tokens(): a LOCAL estimate using tiktoken's cl100k_base. It is not
#     the model's own tokenizer, so treat it as "within ~10%", good for
#     budgeting BEFORE you send a request.
#   * usage_metadata on the response: the provider's EXACT count. Ground truth,
#     but only available AFTER you have already paid for the call.
import tiktoken

_ENC = tiktoken.get_encoding("cl100k_base")


def approx_tokens(text) -> int:
    """Approximate token count for a string (or anything str()-able)."""
    return len(_ENC.encode(str(text)))


# --- Rate-limit-aware calling --------------------------------------------------
# The Groq free tier allows 8000 tokens per minute. Several notebooks here make
# many small calls in a loop, so we self-pace well under the ceiling and retry
# with exponential backoff if we are throttled anyway.

TPM_BUDGET = 3500                       # deliberately conservative
_WINDOW = []                            # [(timestamp, tokens), ...]
USAGE = {"calls": 0, "in": 0, "out": 0, "seconds": 0.0}


def _pace(cost: int):
    """Sleep just enough that our rolling 60s token usage stays under budget."""
    now = time.time()
    while True:
        recent = [(t, n) for (t, n) in _WINDOW if now - t < 60]
        _WINDOW[:] = recent
        if sum(n for _, n in recent) + cost <= TPM_BUDGET or not recent:
            return
        time.sleep(min(5.0, 60 - (now - recent[0][0]) + 0.5))
        now = time.time()


def chat(messages, llm=None, temperature=0.0, max_tokens=300, retries=5):
    """Send `messages`, return the AIMessage. Paces, retries, and meters usage.

    `messages` is a list of (role, content) tuples or LangChain message objects.
    """
    llm = llm or make_llm(temperature=temperature, max_tokens=max_tokens)
    est = approx_tokens(messages) + max_tokens
    delay = 4.0
    for attempt in range(retries):
        _pace(est)
        t0 = time.time()
        try:
            out = llm.invoke(messages)
        except Exception as exc:
            if "rate_limit" in str(exc) or "429" in str(exc):
                time.sleep(delay)
                delay = min(delay * 2, 45)
                continue
            raise
        u = out.usage_metadata or {}
        _WINDOW.append((time.time(), u.get("total_tokens", est)))
        USAGE["calls"] += 1
        USAGE["in"] += u.get("input_tokens", 0)
        USAGE["out"] += u.get("output_tokens", 0)
        USAGE["seconds"] += time.time() - t0
        return out
    raise RuntimeError("still rate limited after %d attempts" % retries)


def ask(prompt: str, system: str = None, **kw) -> str:
    """Convenience wrapper: one user turn in, plain text out."""
    msgs = ([("system", system)] if system else []) + [("user", prompt)]
    return chat(msgs, **kw).content.strip()


print("model:", GROQ_MODEL)
print("key loaded:", bool(os.getenv("GROQ_API_KEY")))
print("tokenizer:", "cl100k_base (approximation)")


### The task set


In [ ]:
# Six multi-step word problems with short, exactly checkable answers. They are
# deliberately the kind that a model usually gets right but sometimes slips on:
# several dependent steps, one trap each (a percentage of the WRONG base, a
# time carry, an off-by-one week, a compounding discount).
#
# Small on purpose. Every technique in this module multiplies the number of
# calls, and the Groq free tier gives us 8000 tokens per minute.

TASKS = [
    dict(id="tank",
         q="A tank holds 480 litres and is currently 3/8 full. You add 90 litres, "
           "then drain an amount equal to 15% of the tank's TOTAL CAPACITY. "
           "How many litres are in the tank now?",
         answer="198"),
    dict(id="train",
         q="A train leaves at 09:40 and travels for 2 hours 50 minutes. You then "
           "wait 35 minutes for a connection, then travel a further 1 hour 15 "
           "minutes. At what time do you arrive? Use 24-hour HH:MM format.",
         answer="14:20"),
    dict(id="book",
         q="A book has 250 pages. Ana reads 12 pages per day on weekdays and 30 "
           "pages per day at weekends. She starts on a Monday morning. On which "
           "day of the week does she finish the book?",
         answer="monday"),
    dict(id="discount",
         q="A jacket costs 200 EUR. A 20% discount is applied, and then a further "
           "15% is taken off the already-reduced price. What is the final price "
           "in EUR?",
         answer="136"),
    dict(id="rect",
         q="A rectangle is three times as long as it is wide. Its perimeter is "
           "64 cm. What is its area in square centimetres?",
         answer="192"),
    dict(id="coins",
         q="You have 17 coins, all of them either 5 cents or 20 cents, worth 205 "
           "cents in total. How many 20-cent coins are there?",
         answer="8"),
]

ANSWER_INSTRUCTION = ("Work through it step by step, briefly. Then give your final "
                      "answer on the last line in exactly this form:\nANSWER: <value>")

import re


def parse_answer(text: str) -> str:
    """Pull the value out of the last 'ANSWER:' line. Returns '' if absent."""
    matches = re.findall(r"ANSWER\s*:\s*(.+)", text, re.I)
    return matches[-1].strip() if matches else ""


def normalise(value: str) -> str:
    """Make answers comparable: lowercase, drop units, currency and separators."""
    v = value.strip().lower()
    v = re.sub(r"\*\*|`|\.$", "", v)
    v = re.sub(r"\b(litres?|liters?|eur|euros?|cm|square centimetres?|cm\^?2|"
               r"cents?|coins?|pages?|hours?)\b", "", v)
    v = v.replace(",", "").replace(" ", "")
    return v.strip()


def is_correct(given: str, expected: str) -> bool:
    g, e = normalise(given), normalise(expected)
    if not g:
        return False
    return g == e or g.endswith(e) or e in g.split("=")[-1]


print("%d tasks" % len(TASKS))
for t in TASKS:
    print("  %-9s expected %-8s | %s..." % (t["id"], t["answer"], t["q"][:56]))


### 1. Draw the samples

The efficient experimental design: draw N samples **once**, then evaluate
majority voting at every smaller N by taking prefixes. This is why the notebook
makes 30 calls instead of 30 + 18 + 6.

Temperature 0.8 - inside the band notebook 01 identified.

In [3]:
from collections import Counter

N_MAX = 5
TEMPERATURE = 0.8

samples = {}                       # task_id -> [answer, answer, ...]
raw = {}                           # task_id -> [full text, ...]
t0 = time.time()

for t in TASKS:
    answers, texts = [], []
    for i in range(N_MAX):
        out = chat([("user", t["q"] + "\n\n" + ANSWER_INSTRUCTION)],
                   temperature=TEMPERATURE, max_tokens=340)
        texts.append(out.content)
        answers.append(parse_answer(out.content))
    samples[t["id"]] = answers
    raw[t["id"]] = texts
    print("%-9s %s" % (t["id"], [normalise(a)[:8] for a in answers]))

print("\n%d calls in %.0f seconds" % (len(TASKS) * N_MAX, time.time() - t0))
print("tokens used so far: %d in, %d out" % (USAGE["in"], USAGE["out"]))

tank      ['198', '198', '198', '198', '198']


train     ['14:20', '15:00', '14:20', '14:20', '14:20']


book      ['', '', '', '', '']


discount  ['136', '136', '136', '136', '136']


rect      ['192', '192', '192', '192', '192']


coins     ['8', '8', '8', '8', '8']

30 calls in 428 seconds
tokens used so far: 2650 in, 7980 out


### 2. The vote

Ten lines. Note that we vote on the **normalised** answer, not the raw string -
"198 litres" and "198" must count as the same vote or the whole thing falls apart.

In [4]:
def majority_vote(answers):
    """Return (winning answer, votes for it, total votes)."""
    normed = [normalise(a) for a in answers if a.strip()]
    if not normed:
        return "", 0, 0
    winner, votes = Counter(normed).most_common(1)[0]
    return winner, votes, len(normed)


print("%-9s %-9s %-9s %8s   %s" % ("task", "expected", "majority", "votes", "all samples"))
print("-" * 74)
for t in TASKS:
    win, votes, total = majority_vote(samples[t["id"]])
    ok = is_correct(win, t["answer"])
    print("%-9s %-9s %-9s %5d/%-2d %s  %s"
          % (t["id"], t["answer"], win[:9], votes, total,
             "OK " if ok else "BAD", [normalise(a)[:7] for a in samples[t["id"]]]))

task      expected  majority     votes   all samples
--------------------------------------------------------------------------
tank      198       198           5/5  OK   ['198', '198', '198', '198', '198']
train     14:20     14:20         4/5  OK   ['14:20', '15:00', '14:20', '14:20', '14:20']
book      monday                  0/0  BAD  ['', '', '', '', '']
discount  136       136           5/5  OK   ['136', '136', '136', '136', '136']
rect      192       192           5/5  OK   ['192', '192', '192', '192', '192']
coins     8         8             5/5  OK   ['8', '8', '8', '8', '8']


### 3. Accuracy vs. N - the measurement

Now the number the technique lives or dies by. For each N we take the first N
samples of each task and vote.

`n=1` is the baseline: a single sample at the same temperature. That is the fair
comparison - not against temperature 0, because you would not run five samples
at temperature 0.

In [5]:
results = []
for n in range(1, N_MAX + 1):
    correct = 0
    for t in TASKS:
        win, _, _ = majority_vote(samples[t["id"]][:n])
        correct += is_correct(win, t["answer"])
    results.append((n, correct, correct / len(TASKS)))

print("%4s %10s %10s %10s" % ("N", "correct", "accuracy", "rel. cost"))
print("-" * 38)
for n, correct, acc in results:
    print("%4d %8d/%-2d %9.0f%% %9.1fx  %s"
          % (n, correct, len(TASKS), acc * 100, n, "#" * int(acc * 24)))

base = results[0][2]
best = max(r[2] for r in results)
print()
print("MEASURED, this run, %s, temperature %.1f, %d tasks:"
      % (GROQ_MODEL, TEMPERATURE, len(TASKS)))
print("  single sample (N=1) : %.0f%%" % (100 * base))
print("  best majority vote  : %.0f%% (at N=%d)"
      % (100 * best, [r[0] for r in results if r[2] == best][0]))
print("  improvement         : %+.0f percentage points for up to %dx the cost"
      % (100 * (best - base), N_MAX))

   N    correct   accuracy  rel. cost
--------------------------------------
   1        5/6         83%       1.0x  ####################
   2        5/6         83%       2.0x  ####################
   3        5/6         83%       3.0x  ####################
   4        5/6         83%       4.0x  ####################
   5        5/6         83%       5.0x  ####################

MEASURED, this run, qwen/qwen3.8-27b, temperature 0.8, 6 tasks:
  single sample (N=1) : 83%
  best majority vote  : 83% (at N=1)
  improvement         : +0 percentage points for up to 5x the cost


In [6]:
if best <= base:
    print("HONEST RESULT: self-consistency did NOT improve accuracy on this task set.")
    print()
    print("That is a real and common outcome, and it has a specific cause: if the")
    print("single-sample accuracy is already at or near 100%, there is no error for")
    print("voting to correct. Voting can only rescue tasks where the model is")
    print("*sometimes* right - it cannot help where it is always right (no upside)")
    print("or always wrong (the wrong answer wins the vote).")
    print()
    print("Do not conclude the technique is useless. Conclude that THIS task set is")
    print("too easy to measure it, which is exactly what a 5x cost increase for +0")
    print("points is telling you.")
else:
    print("Self-consistency improved accuracy by %+.0f points here." % (100 * (best - base)))
    print("Check the per-task table above to see WHICH tasks it rescued - it will")
    print("be the ones where the samples disagreed.")

HONEST RESULT: self-consistency did NOT improve accuracy on this task set.

That is a real and common outcome, and it has a specific cause: if the
single-sample accuracy is already at or near 100%, there is no error for
voting to correct. Voting can only rescue tasks where the model is
*sometimes* right - it cannot help where it is always right (no upside)
or always wrong (the wrong answer wins the vote).

Do not conclude the technique is useless. Conclude that THIS task set is
too easy to measure it, which is exactly what a 5x cost increase for +0
points is telling you.


### 4. The vote spread is a free confidence signal

This is the part people miss, and it is arguably worth more than the accuracy
gain. You already have the samples. The **agreement ratio** tells you how sure
the model is - at no extra cost.

In [7]:
print("%-9s %10s %10s %9s" % ("task", "agreement", "correct?", "reading"))
print("-" * 50)
for t in TASKS:
    win, votes, total = majority_vote(samples[t["id"]])
    ratio = votes / total if total else 0
    ok = is_correct(win, t["answer"])
    reading = ("confident" if ratio >= 0.8 else
               "shaky" if ratio >= 0.5 else "guessing")
    print("%-9s %9.0f%% %10s %9s" % (t["id"], ratio * 100, "yes" if ok else "NO", reading))

print()
print("Use this as a ROUTING signal:")
print("  agreement >= 80%  -> answer directly")
print("  agreement 50-80%  -> answer, but flag uncertainty to the user")
print("  agreement <  50%  -> escalate: bigger model, a tool, or a human")

task       agreement   correct?   reading
--------------------------------------------------
tank            100%        yes confident
train            80%        yes confident
book              0%         NO  guessing
discount        100%        yes confident
rect            100%        yes confident
coins           100%        yes confident

Use this as a ROUTING signal:
  agreement >= 80%  -> answer directly
  agreement 50-80%  -> answer, but flag uncertainty to the user
  agreement <  50%  -> escalate: bigger model, a tool, or a human


> **Pitfall: unanimity is not correctness.** A model that is confidently wrong
> will be *unanimously* wrong - all five paths make the same mistake, because
> they share the same weights and the same misconception. Agreement measures
> stability, not truth. Treat low agreement as a strong negative signal and high
> agreement as a weak positive one.

### 5. When self-consistency cannot work

Four conditions, all necessary:

| Requirement | Why | If violated |
|---|---|---|
| The answer is **extractable and comparable** | You must be able to count votes | An essay has no majority |
| The task has **one right answer** | Voting assumes a mode exists | Creative tasks average to mush |
| Errors are **uncorrelated** | Different paths must fail differently | Systematic bias wins the vote |
| Single-sample accuracy is **between ~30% and ~90%** | Room to improve, and a correct plurality | Too easy = no gain; too hard = confidently wrong |

That last row is the one our measurement just illustrated.

### Weighted voting: not all samples deserve an equal vote. A cheap improvement is


In [ ]:
# to discount samples that failed to produce a parseable answer at all.
def weighted_vote(answers, texts):
    score = Counter()
    for a, txt in zip(answers, texts):
        n = normalise(a)
        if not n:
            continue                       # unparseable: no vote
        w = 1.0
        if len(txt) < 60:                  # suspiciously short reasoning
            w = 0.5
        score[n] += w
    return score.most_common(1)[0] if score else ("", 0)


agree = 0
for t in TASKS:
    plain, _, _ = majority_vote(samples[t["id"]])
    weighted, _ = weighted_vote(samples[t["id"]], raw[t["id"]])
    agree += (plain == weighted)
print("plain and weighted voting agreed on %d/%d tasks." % (agree, len(TASKS)))
print("On a set this small, weighting rarely changes the outcome - it earns its")
print("keep when some samples fail to produce an answer at all.")


### 6. Pitfalls

- **Voting on raw strings.** "198 litres" and "198" must normalise to one vote.
- **Comparing against temperature 0.** The honest baseline is one sample at the
  *same* temperature you sampled at.
- **Treating unanimity as truth.** Correlated errors are unanimous errors.
- **Using it on open-ended output.** There is no mode in a paragraph.
- **Not measuring.** 5x cost for +0 points is a real outcome and you must be
  willing to see it.

### Recap

| Idea | Takeaway |
|---|---|
| Sample N, vote | Ten lines, no framework |
| Normalise before voting | Or your votes never converge |
| Reuse prefixes | Draw N once, evaluate every smaller N for free |
| Agreement ratio | A free confidence signal - use it to route |
| Needs headroom | Helps only when single-sample accuracy is middling |

**Next:** [03_tree_of_thoughts](03_tree_of_thoughts.ipynb) - instead of voting on
finished answers, evaluate partial ones and prune.